# Qlib Research Guide (Google Colab Friendly)This notebook is a kid-friendly ("explain like I'm 5") tour of [Qlib](https://github.com/microsoft/qlib). It walks through installing Qlib, loading data, building models, forecasting, backtesting, and analyzing performance. Every code cell is meant to run top-to-bottom in Google Colab.

## Part 1 — Overview of Qlib- **What is Qlib?** A helpful toolbox that stores lots of stock market numbers so we can teach computers to guess tomorrow's prices. Think of it like a big box of LEGO blocks for finance.- **Why use it?** It saves time for quant researchers by handling data, training models, and testing trading ideas.- **Key words (5-year-old style):**  - **Market data**: daily prices and volumes for many stocks.  - **Features**: numbers we feed to a model (like today's price or the average of last week).  - **Labels**: the answer we want the model to learn (like tomorrow's return).  - **Model**: a smart calculator that learns patterns.  - **Alpha/Signal/Score**: the model's guess about which stocks might go up.  - **Backtesting**: playing a pretend game to see how a strategy would have worked before.

## Part 2 — Installation & SetupRun the next cells in order in Google Colab. They install Qlib, download US data, and initialize the framework.

In [ ]:
# Install from GitHub to get the newest Qlib (works in Colab)!pip install --quiet git+https://github.com/microsoft/qlib.git

In [ ]:
# Imports used throughout the notebookimport osimport pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsimport qlibfrom qlib.constant import REG_USfrom qlib.utils import init_instance_by_configfrom qlib.data.dataset import DatasetHfrom qlib.contrib.data.handler import DataHandlerLPfrom qlib.contrib.model.gbdt import LGBModelfrom qlib.contrib.model.linear import LinearModelfrom qlib.contrib.model.pytorch_gru import GRUModelfrom qlib.contrib.model.pytorch_gats import GATSModelfrom qlib.contrib.strategy import TopkDropoutStrategyfrom qlib.contrib.executor import SimulatorExecutorfrom qlib.contrib.evaluate import backtest, risk_analysisfrom qlib.contrib.report import analysis_position

In [ ]:
# Download US data set to the user directory (~/.qlib/qlib_data)!python -m qlib.run.get_data --target_dir ~/.qlib/qlib_data/us_data --region us --download_mode auto

In [ ]:
# Choose which market to work with (US S&P 500 by default) and initialize QlibMARKET = "sp500"         # U.S. S&P 500 universeBENCHMARK = "SP500"      # Benchmark for backtestsDATA_PATH = os.path.expanduser("~/.qlib/qlib_data/us_data")qlib.init(provider_uri=DATA_PATH, region=REG_US, expression_cache=None, dataset_cache=None)print("Qlib ready! Version:", qlib.__version__)print("Data path exists:", os.path.exists(DATA_PATH))

## Part 3 — Data ProcessingHere we teach Qlib what data to use and how to clean it.- **DataHandler**: a tiny chef that prepares raw price data into tasty features and labels.- **Processors**: the recipe steps (fill missing values, normalize, etc.).- **Dataset**: bundles the handler and the calendar splits (train/valid/test).

In [ ]:
# Configuration for the data handler (feature engineering + labeling)handler_config = {    "class": "DataHandlerLP",    "module_path": "qlib.contrib.data.handler",    "kwargs": {        "start_time": "2017-01-01",        "end_time": "2022-12-31",        "fit_start_time": "2017-01-01",        "fit_end_time": "2020-12-31",        "instruments": MARKET,        # Features: simple price/volume + rolling stats        "feature": [            "$close",            "$open",            "$high",            "$low",            "$volume",            "Ref($close, 1)",            "Ref($close, 5)",            "Mean($close, 5)",            "Mean($close, 20)",            "Std($close, 5)",            "Std($close, 20)",            "Mean($volume, 20)",            "$close / Ref($close, 5) - 1",        ],        # Label: 2-day future return (tomorrow vs. the next day)        "label": ["Ref($close, -2) / Ref($close, -1) - 1"],        # Learn-time processors: drop bad labels, normalize features & labels per day        "learn_processors": [            {"class": "DropnaLabel"},            {"class": "CSRankNorm", "kwargs": {"fields_group": "label"}},            {"class": "RobustZScoreNorm", "kwargs": {"fields_group": "feature"}},            {"class": "Fillna", "kwargs": {"fields_group": "feature"}},        ],        # Infer-time processors: same normalization for inference        "infer_processors": [            {"class": "RobustZScoreNorm", "kwargs": {"fields_group": "feature"}},            {"class": "Fillna", "kwargs": {"fields_group": "feature"}},        ],    },}# Segment the calendar into train/valid/testsegments = {    "train": ("2017-01-01", "2019-12-31"),    "valid": ("2020-01-01", "2020-06-30"),    "test": ("2020-07-01", "2022-12-31"),}# Build the dataset# The DatasetH object glues together the handler recipe and the time segmentsif 'DatasetH' not in globals():    from qlib.data.dataset import DatasetHdataset = DatasetH(handler=handler_config, segments=segments)print(dataset)

In [ ]:
# Load a tidy DataFrame for the training splittrain_df = dataset.prepare("train")valid_df = dataset.prepare("valid")test_df = dataset.prepare("test")print("Train rows:", len(train_df))print(train_df.head())

In [ ]:
# Check the schema: features vs labelsfeature_cols = [c for c in train_df.columns if c.startswith("feature::")]label_cols = [c for c in train_df.columns if c.startswith("label::")]print("Number of features:", len(feature_cols))print("Number of labels:", len(label_cols))

In [ ]:
# Pick one instrument and plot its close price and a label examplesample_inst = train_df.index.get_level_values("instrument")[0]plot_df = train_df.xs(sample_inst, level="instrument")fig, ax1 = plt.subplots(figsize=(10,4))ax1.plot(plot_df.index, plot_df['feature::$close'], label='Close Price', color='blue')ax1.set_ylabel('Price')ax1.legend(loc='upper left')ax1.set_title(f"Sample instrument: {sample_inst}")plt.show()

## Part 4 — ModelingWe try several models from the Qlib model zoo. Hyperparameters are explained simply:- **learning_rate**: how big each learning step is.- **num_leaves / depth**: how big each tree can grow (for LightGBM).- **hidden_size / num_layers**: how many secret neurons and layers (for GRU/GAT).- **drop_prob**: how often we randomly drop units to avoid overfitting.

In [ ]:
# Helper: train a model and get predictions on all splitsdef train_and_predict(model, dataset, name):    print(f"Training {name}...")    model.fit(dataset)    pred = model.predict(dataset)    print(f"{name} done. Prediction sample:", pred.head())    return pred# 1) Linear baselinelinear_model = LinearModel()linear_pred = train_and_predict(linear_model, dataset, "LinearModel")# 2) Gradient-boosted trees (LightGBM)lgb_model = LGBModel(    loss="mse",    learning_rate=0.05,    num_leaves=64,    n_estimators=200,    max_depth=-1,    subsample=0.8,    colsample_bytree=0.8,)lgb_pred = train_and_predict(lgb_model, dataset, "LGBModel")# 3) GRU deep modelgru_model = GRUModel(    d_feat=dataset.handler.get_feature_dim(),    hidden_size=64,    num_layers=2,    drop_prob=0.1,    n_epochs=10,    lr=1e-3,    batch_size=1024,    early_stop=5,    metric="loss",    loss="mse",)gru_pred = train_and_predict(gru_model, dataset, "GRUModel")# 4) Graph Attention Network (GAT)gat_model = GATSModel(    d_feat=dataset.handler.get_feature_dim(),    hidden_size=64,    num_layers=2,    n_epochs=10,    lr=1e-3,    batch_size=512,    drop_prob=0.1,    metric="loss",    loss="mse",)gat_pred = train_and_predict(gat_model, dataset, "GATSModel")

In [ ]:
# Pick one model for the rest of the demobest_pred = lgb_predbest_pred.name = "score"# Extract the aligned label for evaluationlabel = dataset.prepare("test", col_set="label")print(best_pred.head())print(label.head())

## Part 5 — Forecasting / Inference- **Score / Signal / Alpha**: the model's guess of future performance. Bigger score means "more likely to go up" in this simple view.- We save predictions and visualize their distribution.

In [ ]:
# Save predictions to disk (optional)pred_path = "./predictions.csv"best_pred.to_csv(pred_path)print("Saved to", pred_path)# Plot prediction distributionplt.figure(figsize=(6,4))sns.histplot(best_pred.values, bins=40, kde=True)plt.title("Prediction score distribution")plt.show()

## Part 6 — Portfolio & BacktestWe turn scores into trades:- **Strategy**: pick top-K stocks and drop worst ones.- **Executor**: simulates buying/selling with simple assumptions.- **Backtest**: pretend-play to see how money would have grown.- **Metrics (kid style):**  - **Sharpe ratio**: how much good (return) we got for each unit of bumpiness (risk).  - **Drawdown**: the biggest slide from a peak.  - **Turnover**: how often we change our toys (positions).

In [ ]:
# Build strategy from the prediction scoresstrategy_config = {    "class": "TopkDropoutStrategy",    "module_path": "qlib.contrib.strategy",    "kwargs": {        "signal": best_pred,        "topk": 10,        "n_drop": 2,    },}strategy = init_instance_by_config(strategy_config)# Simple simulator executorexecutor_config = {    "class": "SimulatorExecutor",    "module_path": "qlib.contrib.executor",    "kwargs": {        "time_per_step": "day",        "generate_report": True,    },}executor = init_instance_by_config(executor_config)portfolio_metrics, indicator = backtest(    strategy=strategy,    executor=executor,    benchmark=BENCHMARK,    return_type="report",)print("Backtest metrics keys:", portfolio_metrics.keys())

In [ ]:
# Plot equity curve from backtestreport_df = portfolio_metrics['report']plt.figure(figsize=(10,4))plt.plot(report_df['datetime'], report_df['return'].cumsum(), label='Strategy cumulative return')plt.title('Backtest Equity Curve (Pretend Money Growth)')plt.xlabel('Date')plt.ylabel('Cumulative return')plt.legend()plt.show()

In [ ]:
# Compute risk analysis (includes IC & ICIR when signals and labels align)analysis = risk_analysis(report_df, benchmark=BENCHMARK, freq='day')print(analysis)position_report = analysis_position.report_graph(indicator)position_report.head()

## Part 7 — Performance Analysis- **IC (Information Coefficient)**: how well scores line up with future returns (like guessing which toy wins).- **ICIR**: IC stability over time.- **Cumulative/Annualized return**: how money grows total/per year.- **Feature importance**: which inputs mattered most (for tree models).- **Bucket check**: group scores into buckets like a confusion matrix to see performance differences.

In [ ]:
# Feature importance for the LightGBM modelfi = pd.Series(lgb_model.feature_importances_, index=feature_cols).sort_values(ascending=False)fi.head(20).plot(kind='barh', figsize=(6,6))plt.title('Top feature importance (LightGBM)')plt.xlabel('Importance')plt.show()

In [ ]:
# Confusion-matrix-like check: bucket predictions and compare returnspred_label = pd.concat([best_pred.rename('score'), label.iloc[:,0].rename('label')], axis=1).dropna()pred_label['bucket'] = pd.qcut(pred_label['score'], 5, labels=False)bucket_stats = pred_label.groupby('bucket')['label'].mean()print(bucket_stats)bucket_stats.plot(kind='bar', figsize=(6,4))plt.title('Average future return by score bucket')plt.xlabel('Score bucket (low -> high)')plt.ylabel('Mean future return')plt.show()

## Part 8 — Advanced Qlib Features- **Online serving**: save a model and load it later to serve signals.- **Workflow / pipelines**: reuse configs for repeatable experiments.- **Custom pieces**: build your own DataHandler or Model.

In [ ]:
# Save & reload a trained model (serving-style)model_path = "./lgb_model.pkl"lgb_model.save(model_path)print("Saved model to", model_path)# Load backreloaded = LGBModel()reloaded.load(model_path)print("Reloaded model ready:", reloaded)

In [ ]:
# Hyperparameter search example (very small for demo)search_spaces = {    "learning_rate": [0.01, 0.05],    "num_leaves": [31, 63],}best_params = Nonebest_ic = -np.inffor lr in search_spaces['learning_rate']:    for nl in search_spaces['num_leaves']:        temp_model = LGBModel(learning_rate=lr, num_leaves=nl, n_estimators=50)        temp_model.fit(dataset)        pred = temp_model.predict(dataset)        joined = pd.concat([pred.rename('score'), label.iloc[:,0].rename('label')], axis=1).dropna()        ic = joined[['score','label']].corr().iloc[0,1]        if ic > best_ic:            best_ic = ic            best_params = {"learning_rate": lr, "num_leaves": nl}print("Best params (tiny search):", best_params, "IC=", best_ic)

In [ ]:
# Template for a custom DataHandlerfrom qlib.data.dataset.handler import DataHandlerLP as BaseHandlerclass MyCustomHandler(BaseHandler):    def get_feature_config(self):        return ["$close", "Mean($close, 5)"]    def get_label_config(self):        return ["Ref($close, -1) / $close - 1"]# Example creation (not used elsewhere in the notebook)custom_dataset = DatasetH(handler={"class": "MyCustomHandler", "module_path": "__main__"}, segments=segments)print("Custom handler dataset ready")

In [ ]:
# Template for a custom modelfrom qlib.model.base import Modelclass MyTinyModel(Model):    def fit(self, dataset):        # pretend to train; normally you would use dataset.prepare()        self.mean_label = dataset.prepare("train", col_set="label").mean().values[0]    def predict(self, dataset):        # predict the same value for all rows (toy example)        label_index = dataset.prepare("train", col_set="label").index        return pd.Series(self.mean_label, index=dataset.prepare("train").index)print("Custom model template defined")

## Part 9 — Summary & Next Steps- Try more features (fundamentals, news) and more models.- Explore [Qlib documentation](https://qlib.readthedocs.io/) and examples.- Contribute fixes or new models to the [Qlib repo](https://github.com/microsoft/qlib).- Test other datasets (e.g., US full market) by switching the `MARKET` and download target.